In [ ]:
import sqlite3

import pandas as pd
import ollama
from tqdm import tqdm

from unga81.config import DATABASE_PATH
from unga81.database import create_database

In [ ]:
create_database()

In [ ]:
def get_all_countries(conn: sqlite3.Connection) -> pd.DataFrame:
    """Return all the countries.

    Args:
        conn (sqlite3.Connection): Database connection.

    Returns:
        pd.DataFrame: Data with countries.
    """
    query = """select country, iso_3, url, full_speech, wordcount from country;"""
    with conn:
        cursor = conn.cursor()
        cursor.execute(query)
        rows = cursor.fetchall()
        column_names = [x[0] for x in cursor.description]
        cursor.close()

    df = pd.DataFrame(rows, columns=column_names)
    return df


conn = sqlite3.connect(DATABASE_PATH)
df = get_all_countries(conn=conn)
conn.close()

In [ ]:
df.info()

In [ ]:
summary_prompt = """Analyze this presidential speech text and create a 2-3 sentence summary capturing the main points and conclusions. Focus on:
- Primary objectives or goals stated
- Key arguments or positions made
- Overall conclusion or call to action
Output ONLY the summary, nothing else.

Speech:
'''{text}'''
"""

countries_mentioned_prompt = """Extract all countries mentioned in this text in exactly what order they appear. 
Format as a comma-separated list with country names only (e.g., "France, Japan, Germany").
Ignore proper nouns that are not standalone countries (cities like Paris don't count).
Output ONLY the ordered list, nothing else.

Text:
'''{text}'''
"""

risks_prompt = """Identify all explicit and implicit risks or negative consequences mentioned in this text.
- Economic threats or market concerns
- Geopolitical conflicts or instability
- Social/environmental dangers or crises
- Policy failures or implementation challenges

For each risk, briefly state it AND its source/cause. Do not invent risks beyond what is implied or stated.
Output ONLY one of these: "no_risks_found" OR "risk_1, risk_2..." with brief descriptions.

Text:
'''{text}'''
"""

haiku_prompt = """Craft a 3-line haiku inspired by the key themes or messages of the speech. Reply only with the lines of the haiku.
Avoid inventing or paraphrasing concepts not present in the text. Output ONLY the haiku, nothing else.

Speech:
'''{text}'''
"""

single_word_prompt = """Extract a single word that best represents this entire presidential speech's core message or main point. 
The word should be one the speaker themselves might use when asking "What is THIS about?"
Output ONLY the single word, nothing else.

Speech:
'''{text}'''
"""

hashtags_prompt = """Create a set of relevant hashtags representing this presidential speech's content and themes. 

Requirements:
- Include 5-7 hashtags total
- Prioritize topics that would appear on Twitter/X when discussing this speech
- Mix broad topic tags with more specific keywords 
- Use #hashtags only (no spaces)

Output ONLY the hashtag list, nothing else.

Example output:
#PresidentialSpeech #ClimatePolicy #InternationalRelations #GlobalCooperation #NewAdministration #EconomicGrowth #ForeignPolicyMatters #VotersFirst

Speech:
'''{text}'''
"""

prompt_headlines = """Generate a single impactful newspaper headline for each one of the following styles: 
[leftist, alt-right, neo-liberal, sensationalist, nationalist, environmental, financial, sports] from the following speech. 
Each headline should be unique and accurately reflect the content of the speech. Do not include the name of the paper.

The output should be as follows. Output ONLY the headlines. Nothing else:
- [leftist] <leftist headline>
- [alt-right] <alt-right headline>
- [neo-liberal] <neo-liberal headline>
- [sensationalist] <sensationalist headline>
- [environmental] <environmental headline>
- [financial] <financial headline>
- [sports] <sports headline>

Speech: 
'''{text}'''"""
prompts = [
    ("summary", summary_prompt),
    ("countries_mentioned", countries_mentioned_prompt),
    ("risks", risks_prompt),
    ("haiku", haiku_prompt),
    ("single_word", single_word_prompt),
    ("hashtags", hashtags_prompt),
    ("headlines", prompt_headlines),
]

In [ ]:
conn = sqlite3.connect(DATABASE_PATH)

In [ ]:
def insert_into_analysis(conn: sqlite3.Connection, data: dict):
    query_delete = """delete from analysis where country = ?;"""
    query_insert = """INSERT INTO analysis
                    (country,
                     iso_3,
                     summary,
                     countries_mentioned,
                     risks,
                     haiku,
                     single_word,
                     hashtags,
                     headlines)
                    VALUES(?, ?, ?, ?, ?, ?, ?, ?, ?);"""
    with conn:
        cursor = conn.cursor()
        cursor.execute(query_delete, (data["country"],))
        conn.commit()

        cursor.execute(
            query_insert,
            (
                data["country"],
                data["iso_3"],
                data["summary"],
                data["countries_mentioned"],
                data["risks"],
                data["haiku"],
                data["single_word"],
                data["hashtags"],
                data["headlines"],
            ),
        )
        conn.commit()

In [ ]:
def update_prompt_result(prompt: str, model: str): ...

In [ ]:
llm = "llama3.2"
for i, r in df.iterrows():
    data = {}
    data["country"] = r["country"]
    data["iso_3"] = r["iso_3"]
    pbar = tqdm(prompts)
    for name, prompt in pbar:
        pbar.set_description(f"{i+1}/{df.shape[0]} {r["country"]} - {name}")
        result = ollama.generate(
            model=llm,
            prompt=prompt.format(text=r["full_speech"]),
            think=False,
        )
        data[name] = str(result["response"])
    insert_into_analysis(conn=conn, data=data)

In [ ]:
prompt_headlines = """Generate a single impactful newspaper headline for each one of the following styles: 
[leftist, alt-right, neo-liberal, sensationalist, nationalist, environmental, financial, sports] from the following speech. 
Each headline should be unique and accurately reflect the content and tone of the speech. Do not include the name of the paper.

The output should be as follows. Output ONLY the headlines. Nothing else:
- [leftist] <leftist headline>
- [alt-right] <alt-right headline>
- [neo-liberal] <neo-liberal headline>
- [sensationalist] <sensationalist headline>
- [environmental] <environmental headline>
- [financial] <financial headline>
- [sports] <sports headline>

Speech: 
'''{text}'''"""
result = ollama.generate(
    model="llama3.2",
    prompt=prompt_headlines.format(text=text),
    think=False,
)
print(result["response"])